In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [2]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=10", con=connection3)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=10"

c1= text(query)
connection3.execute(c1)

2023-08-23


In [3]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMPAC10 WHERE pacfilfec >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection2)
connection2.close()



In [4]:
base2.shape

(174793, 46)

In [5]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174793 entries, 0 to 174792
Data columns (total 46 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   oricenasicod      174793 non-null  object        
 1   cenasicod         174793 non-null  object        
 2   pacsecnum         174793 non-null  int64         
 3   pachisclinum      174793 non-null  int64         
 4   tipopacicod       174793 non-null  object        
 5   pachiscliubi      140424 non-null  object        
 6   esthisclicod      174793 non-null  object        
 7   pacfilfec         174793 non-null  object        
 8   pachispasfec      88 non-null      datetime64[ns]
 9   pachisdepfec      0 non-null       object        
 10  pacfallfec        270 non-null     datetime64[ns]
 11  pacderatenext     174793 non-null  object        
 12  pacderatenextfec  0 non-null       object        
 13  pacrefflg         174793 non-null  object        
 14  motb

In [6]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

base2.to_sql(name='tmp_cmpac10', con=engine3, if_exists='replace', index=False)


793

In [7]:

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmpac10 FALSO CON LO OBTENIDO DEL ORACLE


borrando=f"DELETE FROM cmpac10 WHERE pacfilfec >='{fecha}'"
borrado = connection3.execute(borrando)

query_count_before = "SELECT COUNT(*) FROM cmpac10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cmpac10 antes de la inserción: {cant_antes}")


query="""

ALTER TABLE tmp_cmpac10 
ALTER COLUMN oricenasicod  TYPE character(1),
ALTER COLUMN cenasicod  TYPE character(3),
ALTER COLUMN pacsecnum  TYPE numeric(10,0),
ALTER COLUMN pachisclinum  TYPE numeric(10,0),
ALTER COLUMN tipopacicod  TYPE character(1),
ALTER COLUMN pachiscliubi  TYPE character(12),
ALTER COLUMN esthisclicod  TYPE character(1),
ALTER COLUMN pacfilfec  TYPE date,
ALTER COLUMN pachispasfec TYPE date USING pachispasfec::date,
ALTER COLUMN pachisdepfec TYPE date USING pachisdepfec::date,
ALTER COLUMN pacfallfec TYPE date USING pacfallfec::date,
ALTER COLUMN pacderatenext  TYPE character(1),
ALTER COLUMN pacderatenextfec TYPE date USING pacderatenextfec::date,
ALTER COLUMN pacrefflg  TYPE character(1),
ALTER COLUMN motblohisclicod  TYPE character(3),
ALTER COLUMN pacblofecdes TYPE date USING pacblofecdes::date,
ALTER COLUMN pacblofechas TYPE date USING pacblofechas::date,
ALTER COLUMN pacobs  TYPE character(100),
ALTER COLUMN pachisanufec TYPE date USING pachisanufec::date,
ALTER COLUMN pachisactfec TYPE date USING pachisactfec::date,
ALTER COLUMN pacfamnom  TYPE character(30),
ALTER COLUMN pacfamdir  TYPE character(100),
ALTER COLUMN pacfamtel  TYPE character(10),
ALTER COLUMN pactelfij  TYPE character(10),
ALTER COLUMN pactelcel  TYPE character(10),
ALTER COLUMN pacmail  TYPE character(100),
ALTER COLUMN pacestpercenano  TYPE numeric(4,0),
ALTER COLUMN pacestpercencod  TYPE character(1),
ALTER COLUMN pacestregcod  TYPE character(1),
ALTER COLUMN pacusucrecod  TYPE character(10),
ALTER COLUMN paccrefec TYPE date USING paccrefec::date,
ALTER COLUMN pacusumodcod  TYPE character(10),
ALTER COLUMN pacmodfec TYPE date USING pacmodfec::date,
ALTER COLUMN convcod  TYPE character(3),
ALTER COLUMN pacadsdepinifec TYPE date USING pacadsdepinifec::date,
ALTER COLUMN pacadsdepfinfec TYPE date USING pacadsdepfinfec::date,
ALTER COLUMN opertelfcod  TYPE character(2),
ALTER COLUMN tipiafacod  TYPE character(3),
ALTER COLUMN pacflgmovhc  TYPE character(1),
ALTER COLUMN pacfamubigeo  TYPE character(6),
ALTER COLUMN tipviacod  TYPE character(2),
ALTER COLUMN paccoordx  TYPE numeric(20,10),
ALTER COLUMN paccoordy  TYPE numeric(20,10),
ALTER COLUMN pacrefdir  TYPE character(100),
ALTER COLUMN paccitasinfil  TYPE character(1),
ALTER COLUMN pacdonaflg TYPE numeric(1,0) USING pacdonaflg::numeric(1,0);




INSERT INTO cmpac10 
(oricenasicod,cenasicod,pacsecnum,pachisclinum,tipopacicod,pachiscliubi,esthisclicod,pacfilfec,pachispasfec,pachisdepfec,pacfallfec,pacderatenext,pacderatenextfec,pacrefflg,motblohisclicod,pacblofecdes,pacblofechas,pacobs,pachisanufec,pachisactfec,pacfamnom,pacfamdir,pacfamtel,pactelfij,pactelcel,pacmail,pacestpercenano,pacestpercencod,pacestregcod,pacusucrecod,paccrefec,pacusumodcod,pacmodfec,convcod,pacadsdepinifec,pacadsdepfinfec,opertelfcod,tipiafacod,pacflgmovhc,pacfamubigeo,tipviacod,paccoordx,paccoordy,pacrefdir,paccitasinfil,pacdonaflg) 

SELECT 
oricenasicod,cenasicod,pacsecnum,pachisclinum,tipopacicod,pachiscliubi,esthisclicod,pacfilfec,pachispasfec,pachisdepfec,pacfallfec,pacderatenext,pacderatenextfec,pacrefflg,motblohisclicod,pacblofecdes,pacblofechas,pacobs,pachisanufec,pachisactfec,pacfamnom,pacfamdir,pacfamtel,pactelfij,pactelcel,pacmail,pacestpercenano,pacestpercencod,pacestregcod,pacusucrecod,paccrefec,pacusumodcod,pacmodfec,convcod,pacadsdepinifec,pacadsdepfinfec,opertelfcod,tipiafacod,pacflgmovhc,pacfamubigeo,tipviacod,paccoordx,paccoordy,pacrefdir,paccitasinfil,pacdonaflg

FROM tmp_cmpac10 
;
"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmpac10;
SELECT COUNT(*) FROM cmpac10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla cmpac10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection3.close() 


Cantidad de filas en la tabla cmpac10 antes de la inserción: 36338374
Cantidad de filas en la tabla cmpac10 después de la inserción: 36513167
La cantidad de filas insertadas fue de: 174793
